# ChagaSight — Fold Training Notebook v10

## Changes vs v9
| # | What changed | Why |
|---|---|---|
| 1 | `BATCH_SIZE 32 → 16` | 6 GB GPU cannot sustain Phase 2 at batch 32 with 173 M params unfrozen; was running at 0.09 it/s instead of ~0.6 it/s |
| 2 | `phase2_grad_accum 1 → 2` | Restores effective batch 32 in Phase 2 while fitting in VRAM |
| 3 | GPU memory check cell added | Catches VRAM pressure before committing to Phase 2 |
| 4 | Removed hardcoded eff.batch comment `16×GRAD_ACCUM` | Trainer display was wrong when BATCH_SIZE≠16; now shows actual values |
| 5 | `NUM_WORKERS 4 → 2` | Fewer workers reduce memory pressure on 6 GB GPU |
| 6 | Removed misleading Phase-1-is-blind "NOTE" block | Already documented in trainer; duplicate text was confusing |
| 7 | `val_every_n_iters` set to `VAL_EVERY_P2` only (no re-override after init) | Previous notebook set it twice, creating confusion |
| 8 | Checkpoint verification cell | Confirms saved file sizes before proceeding to next fold |

## Workflow
```
pretrain ST-MEM → stmem_1d_pretrained.pt
pretrain MAE    → mae_2d_pretrained.pt
this notebook   → fold{N}_best.pt  (repeat FOLD = 0..4)
evaluation      → evaluation_complete_final.ipynb
```

**Expected times on RTX 3050 6 GB:**
- Phase 1: ~18 min (2 000 iters, batch 16, accum 4, FM frozen)
- Phase 2: ~5–6 h  (12 000 iters, batch 16, accum 2, full model)


## Cell 1 — Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders
from src.training.trainer import ChagasTrainer

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'GPU required — switch runtime to GPU'

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({gpu_gb:.1f} GB)')
assert gpu_gb >= 5.5, f'Need ≥6 GB GPU, found {gpu_gb:.1f} GB'


## Cell 2 — Configuration

In [ ]:
# ── QUICK TEST ─────────────────────────────────────────────────────────────
# True  → 5-min smoke-test (loss must decrease, no crashes; score meaningless)
# False → full training (~5-6 h on RTX 3050 6 GB)
QUICK_TEST = False

# ── FOLD ───────────────────────────────────────────────────────────────────
FOLD = 0   # change to 1, 2, 3, 4 for other folds

# ── PATHS ──────────────────────────────────────────────────────────────────
DATA_DIR     = project_root / 'data' / 'processed'
METADATA_CSV = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR   = DATA_DIR / '2d_images'
SIGNALS_DIR  = DATA_DIR / '1d_signals_100hz'
CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MAE_CHECKPOINT   = CHECKPOINT_DIR / 'mae_2d_pretrained.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained.pt'

assert METADATA_CSV.exists(), f'Missing: {METADATA_CSV}'
assert MAE_CHECKPOINT.exists(),   f'Missing: {MAE_CHECKPOINT} — run MAE pretraining first'
assert STMEM_CHECKPOINT.exists(), f'Missing: {STMEM_CHECKPOINT} — run ST-MEM pretraining first'

# ── HARDWARE (corrected for 6 GB GPU) ──────────────────────────────────────
# BATCH_SIZE=16 is required for Phase 2 on a 6 GB GPU.
# phase1_grad_accum=4  → eff. batch = 16×4 = 64  (matches Van Santvliet paper)
# phase2_grad_accum=2  → eff. batch = 16×2 = 32  (halved to fit 173 M params)
BATCH_SIZE          = 16   # DO NOT increase to 32 on 6 GB GPU for Phase 2
PHASE1_GRAD_ACCUM   = 4    # eff.batch = 64
PHASE2_GRAD_ACCUM   = 2    # eff.batch = 32  (was 1 in v9 — caused OOM slowdown)
NUM_WORKERS         = 2
USE_AMP             = True

# ── TRAINING SCHEDULE (Van Santvliet et al. 2025) ──────────────────────────
PHASE1_ITERATIONS = 2000
PHASE2_ITERATIONS = 12000
PHASE1_LR         = 2e-4
PHASE2_LR_HIGH    = 2e-4
PHASE2_LR_LOW     = 2e-5
MAX_GRAD_NORM     = 1.0
WARMUP_ITERS      = 200

# ── VALIDATION ─────────────────────────────────────────────────────────────
# Phase 1 → 0 mid-phase checks (2000 / 4000 = 0; blind, FM frozen, acceptable)
# Phase 2 → 3 checks at iters 4000, 8000, 12000
VAL_EVERY = 4000   # single value used globally; see trainer note below

# ── QUICK-TEST OVERRIDES ───────────────────────────────────────────────────
if QUICK_TEST:
    PHASE1_ITERATIONS = 25
    PHASE2_ITERATIONS = 50
    WARMUP_ITERS      = 10
    VAL_EVERY         = 50
    NUM_WORKERS       = 0

# ── AUTO-RESUME ────────────────────────────────────────────────────────────
_ckpt = CHECKPOINT_DIR / f'fold{FOLD}_latest.pt'
RESUME_FROM = str(_ckpt) if _ckpt.exists() else None

# ── SUMMARY ────────────────────────────────────────────────────────────────
print(f'Fold {FOLD} | QUICK_TEST={QUICK_TEST}')
print(f'Batch {BATCH_SIZE} | P1 eff.batch={BATCH_SIZE*PHASE1_GRAD_ACCUM} | P2 eff.batch={BATCH_SIZE*PHASE2_GRAD_ACCUM}')
print(f'Phase 1: {PHASE1_ITERATIONS} iters | Phase 2: {PHASE2_ITERATIONS} iters | Val every {VAL_EVERY}')
print(f'Resume: {RESUME_FROM or "fresh start"}')


## Cell 3 — GPU Memory Check

In [ ]:
# Verify GPU has enough free memory before loading the full 173 M-param model.
torch.cuda.empty_cache()
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb  = torch.cuda.memory_reserved()  / 1e9
total_gb     = torch.cuda.get_device_properties(0).total_memory / 1e9
free_gb      = total_gb - reserved_gb

print(f'GPU memory:  {total_gb:.1f} GB total | {allocated_gb:.2f} GB allocated | {free_gb:.2f} GB free')
if free_gb < 3.5:
    raise RuntimeError(f'Insufficient GPU memory: {free_gb:.1f} GB free, need ≥3.5 GB. Restart kernel.')
print('Memory check passed.')


## Cell 4 — Dataloaders

In [ ]:
train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,
    augment_train=True,
)

# Sanity check first batch
batch = next(iter(train_loader))
assert batch['image'].shape == torch.Size([BATCH_SIZE, 3, 24, 2048]), 'Image shape wrong'
assert batch['signal'].shape == torch.Size([BATCH_SIZE, 12, 1000]),   'Signal shape wrong'
assert not torch.isnan(batch['signal']).any(), 'NaN in signals'

unique_labels = sorted(set(round(v, 2) for v in batch['label'].tolist()))
assert set(unique_labels) <= {0.0, 0.2, 0.8, 1.0}, f'Unexpected labels: {unique_labels}'
print(f'Train {len(train_loader.dataset):,} samples | Val {len(val_loader.dataset):,} samples')
print(f'Label values in first batch: {unique_labels}')


## Cell 5 — Model + Pretrained Weights

In [ ]:
model = HybridChagasModel(
    img_size=(24, 2048), patch_size_2d=(8, 64),
    num_leads=12, seq_len_1d=1000, patch_size_1d=50,
    embed_dim=768, depth=12, num_heads=12,
    use_aol=True, use_demographics=True,
)

model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))
model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))
model = model.to(device)

# Forward-pass sanity check
with torch.no_grad():
    out = model(batch['image'].to(device), batch['signal'].to(device),
                batch['age'].to(device),   batch['sex'].to(device))
assert torch.isfinite(out['logits']).all(),       'Non-finite logits at init'
assert torch.isfinite(out['fm_features']).all(),  'Non-finite FM features at init'

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_p:,} | Trainable: {trainable_p:,}')
print(f'Logits shape: {out["logits"].shape} | FM features: {out["fm_features"].shape}')


## Cell 6 — Trainer

In [ ]:
trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    phase1_iterations=PHASE1_ITERATIONS,
    phase2_iterations=PHASE2_ITERATIONS,
    phase1_lr=PHASE1_LR,
    phase2_lr_high=PHASE2_LR_HIGH,
    phase2_lr_low=PHASE2_LR_LOW,
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_iters=WARMUP_ITERS,
    phase1_grad_accum=PHASE1_GRAD_ACCUM,
    phase2_grad_accum=PHASE2_GRAD_ACCUM,
    val_every_n_iters=VAL_EVERY,
    val_subset_size=300 if QUICK_TEST else 3000,
    val_n_permutations=100 if QUICK_TEST else 1000,
)
# Single val_every_n_iters applies to both phases:
#   Phase 1: 2000 / 4000 = 0 checks  (blind — FM frozen, short, low risk)
#   Phase 2: 12000 / 4000 = 3 checks (at iters 4000, 8000, 12000)
trainer.val_every_n_iters = VAL_EVERY

p2_checks = PHASE2_ITERATIONS // VAL_EVERY
print(f'Val interval: {VAL_EVERY} iters | Phase 2 mid-checks: {p2_checks}')
print(f'Val subset: {trainer.val_subset_size} stratified samples | Perms(fast): {trainer.val_n_permutations}')
if QUICK_TEST:
    print('QUICK_TEST mode — expected time ~5 min')
else:
    print('Full training — ETA Phase 1 ~18 min | Phase 2 ~5-6 h')


## Cell 7 — Train

In [ ]:
if RESUME_FROM:
    print(f'Resuming from: {Path(RESUME_FROM).name}')
else:
    print('Starting fresh')

metrics = trainer.train(fold=FOLD, resume_from=RESUME_FROM)

print(f'\nFold {FOLD} final results:')
print(f'  TPR@5%: {metrics["tpr_5pct"]:.4f}  (PRIMARY)')
print(f'  AUROC:  {metrics["auroc"]:.4f}')
print(f'  AUPRC:  {metrics.get("auprc", 0):.4f}')
print(f'  Method: {"OFFICIAL" if metrics.get("using_official") else "APPROXIMATE"}')

if not QUICK_TEST:
    tpr = metrics['tpr_5pct']
    if tpr >= 0.445:
        print('Matches/beats top team (0.445)')
    elif tpr >= 0.420:
        print(f'Target achieved (≥0.42) | Gap to top team: {0.445 - tpr:.4f}')
    elif tpr >= 0.35:
        print(f'Below target | Gap to 0.42: {0.42 - tpr:.4f}')
    else:
        print('Score low — verify pretraining weights loaded correctly')


## Cell 8 — Save Results & Training Curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Save CSV
results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_df['quick_test'] = QUICK_TEST
results_df.to_csv(CHECKPOINT_DIR / f'fold{FOLD}_results.csv', index=False)

history = trainer.history
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# ── Loss curve ─────────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
if history['train_loss']:
    ax0.plot(history['train_loss'], lw=0.8, alpha=0.8, color='steelblue', label='Train loss (50-iter smooth)')
    if PHASE1_ITERATIONS < len(history['train_loss']):
        ax0.axvline(PHASE1_ITERATIONS, color='r', ls='--', lw=1, label=f'Phase 2 start ({PHASE1_ITERATIONS})')
    ax0.set_xlabel('Iteration')
    ax0.set_ylabel('Loss')
    ax0.set_title('Training Loss')
    ax0.legend(fontsize=8)
    ax0.grid(True, alpha=0.3)

# ── TPR@5% curve ───────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
if history['val_tpr_5pct']:
    iters = [VAL_EVERY * (i + 1) for i in range(len(history['val_tpr_5pct']))]
    ax1.plot(iters, history['val_tpr_5pct'], 'go-', ms=5, lw=1.5, label='Val TPR@5%')
    ax1.axhline(0.42,  color='r',      ls='--', lw=1, alpha=0.8, label='Target 0.420')
    ax1.axhline(0.445, color='purple', ls=':',  lw=1, alpha=0.8, label='Top team 0.445')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('TPR@5%')
    ax1.set_title('Validation TPR@5% (primary metric)')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, min(1.0, max(history['val_tpr_5pct']) * 1.15))

# ── Gradient norm ──────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[2])
if history['grad_norm']:
    g = history['grad_norm']
    n_show = min(PHASE1_ITERATIONS, len(g))
    ax2.plot(g[:n_show], lw=0.6, alpha=0.6, color='darkorange')
    ax2.axhline(1.0, color='r', ls='--', lw=1, label='Clip @ 1.0')
    clipped_pct = 100 * sum(1 for v in g[:n_show] if v > 1.0) / max(1, n_show)
    ax2.set_title(f'Gradient Norm — Phase 1 ({clipped_pct:.0f}% clipped)')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Grad norm')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

fig.suptitle(f'Fold {FOLD} Training{"  [QUICK TEST]" if QUICK_TEST else ""}', fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = CHECKPOINT_DIR / f'fold{FOLD}_training_curve.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')


## Cell 9 — Checkpoint Verification

In [ ]:
print(f'Checkpoints for fold {FOLD}:')
for p in sorted(CHECKPOINT_DIR.glob(f'fold{FOLD}*.pt')):
    mb = p.stat().st_size / 1e6
    print(f'  {p.name}  ({mb:.0f} MB)')

# Quick sanity-load best checkpoint
best_path = CHECKPOINT_DIR / f'fold{FOLD}_best.pt'
if best_path.exists():
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    saved_score = ckpt.get('val_score', 0.0)
    saved_phase = ckpt.get('phase', '?')
    saved_iter  = ckpt.get('iteration', '?')
    print(f'\nbest checkpoint: val_score={saved_score:.4f}  phase={saved_phase}  iter={saved_iter}')
    del ckpt

if not QUICK_TEST:
    print('\nNext steps:')
    for f in range(5):
        status = 'DONE' if (CHECKPOINT_DIR / f'fold{f}_best.pt').exists() else 'pending'
        marker = '✓' if status == 'DONE' else ' '
        print(f'  [{marker}] Fold {f} — {status}')
    completed = sum(1 for f in range(5) if (CHECKPOINT_DIR / f'fold{f}_best.pt').exists())
    if completed == 5:
        print('\nAll 5 folds complete — run evaluation_complete_final.ipynb')
